<a href="https://colab.research.google.com/github/SELOMELO280104/aware_bitki/blob/main/Aware_Bitki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
!mkdir yolov8_data
!unzip -q Bitki.v1i.yolov8.zip -d yolov8_data/

from ultralytics import YOLO

model_v8 = YOLO('yolov8n-seg.pt')
model_v8.train(data='yolov8_data/data.yaml', epochs=50, imgsz=640)

In [ ]:
!mkdir yolov11_data
!unzip -q Bitki.v1i.yolov11.zip -d yolov11_data/

from ultralytics import YOLO

model_v11 = YOLO('yolo11n-seg.pt')
model_v11.train(data='yolov11_data/data.yaml', epochs=50, imgsz=640)

In [ ]:
# Eğer daha önce klasör açıldıysa hata vermemesi için -p ekledik
!mkdir -p yolov26_data

# Dosyaları üstüne yazarak (-o) çıkarıyoruz ki uyarı vermesin
!unzip -q -o Bitki.v1i.yolo26.zip -d yolov26_data/

from ultralytics import YOLO

# Dosya adını tam olarak sol tarafta yüklü olan yolo26n.pt yaptık
model_v26 = YOLO('yolo26n.pt')
model_v26.train(data='yolov26_data/data.yaml', epochs=50, imgsz=640)

Sonuçlar

In [ ]:
import os
import shutil
from google.colab import files

# Ana klasör ve alt klasörleri oluşturuyoruz
ana_klasor = "proje_dosyalari"
modeller_klasoru = os.path.join(ana_klasor, "modeller")
sonuclar_klasoru = os.path.join(ana_klasor, "sonuclar")

os.makedirs(modeller_klasoru, exist_ok=True)
os.makedirs(sonuclar_klasoru, exist_ok=True)

# Hangi klasörün hangi modele ait olduğunu koda söylüyoruz
klasor_model_eslesmesi = {
    os.path.join("runs", "segment", "train"): "YOLOv8",
    os.path.join("runs", "segment", "train2"): "YOLOv11",
    os.path.join("runs", "detect", "train"): "YOLOv26"
}

# Almak istediğimiz önemli dosyalar
istenen_dosyalar = ["results.png", "confusion_matrix.png", "weights/best.pt"]

for klasor, model_adi in klasor_model_eslesmesi.items():
    if os.path.exists(klasor):
        for dosya in istenen_dosyalar:
            dosya_yolu = os.path.join(klasor, dosya)

            if os.path.exists(dosya_yolu):
                # Model dosyalarını isimlendirip "modeller" klasörüne atıyoruz
                if "best.pt" in dosya:
                    yeni_ad = f"{model_adi}_modeli.pt"
                    hedef_yol = os.path.join(modeller_klasoru, yeni_ad)

                # Grafikleri isimlendirip "sonuclar" klasörüne atıyoruz
                else:
                    yeni_ad = f"{model_adi}_{os.path.basename(dosya)}"
                    hedef_yol = os.path.join(sonuclar_klasoru, yeni_ad)

                shutil.copy(dosya_yolu, hedef_yol)

# Dosyaları zip yapıp indiriyoruz
zip_adi = "derli_toplu_sunum_dosyalari"
shutil.make_archive(zip_adi, 'zip', ana_klasor)
files.download(f"{zip_adi}.zip")

print("Dosyalar alt klasörlere ayrıldı, isimlendirildi ve indirme başlatıldı!")

In [ ]:
import pandas as pd
import os

# Modellerin sonuç dosyalarının (CSV) yolları
csv_yollari = {
    "YOLOv8 (Maskeleme)": "runs/segment/train/results.csv",
    "YOLOv11 (Maskeleme)": "runs/segment/train2/results.csv",
    "YOLO26 (Sadece Kutu)": "runs/detect/train/results.csv"
}

for model_adi, yol in csv_yollari.items():
    if os.path.exists(yol):
        # CSV dosyasını oku ve sütun adlarındaki gereksiz boşlukları temizle
        df = pd.read_csv(yol)
        df.columns = df.columns.str.strip()

        # En son satırı (50. epoch sonuçlarını) al
        son_sonuc = df.iloc[-1]

        print(f"--- {model_adi} Sonuçları ---")

        # Segmentasyon (Maskeleme) modelleri için maske skorlarını yazdır
        if 'metrics/mAP50(M)' in df.columns:
            print(f"Maskeleme (mAP50): {son_sonuc['metrics/mAP50(M)']:.4f}")
            print(f"Maskeleme (mAP50-95): {son_sonuc['metrics/mAP50-95(M)']:.4f}")

        # Tespit (Kutu) skorlarını yazdır
        print(f"Nesne Tespiti Kutu (mAP50): {son_sonuc['metrics/mAP50(B)']:.4f}")
        print("-" * 30 + "\n")
    else:
        print(f"{model_adi} için results.csv bulunamadı.\n")